# Chapter 04-06 · Preprocessing: imputation, encoding, scaling, transforms

**Label:** Core  |  **Time:** ~55 minutes  |  **Difficulty:** moderate

**Prerequisites:** 04-05 for the leakage constraint, 03-07 for scaling, 02-04 for missing values.

**Position in the learning path:** module 04, chapter 6 of 8.

---

## Why this matters

04-05 treated preprocessing as a hazard. This chapter treats it as a subject.

Every real dataset arrives with holes, with words where numbers are expected, with columns measured in
incompatible units, and with distributions that are the wrong shape. **Something has to be done about all
of it before a model can be fitted, and every one of those somethings is a modelling decision** - it
changes the answer, it encodes an assumption, and it can be wrong.

The four decisions, and the questions they turn on:

| Decision | The question it answers |
|---|---|
| **Imputation** | what do I put where a value is missing - and does the missingness itself mean something? |
| **Encoding** | how does a word become a number without inventing an order that is not there? |
| **Scaling** | do my columns need to be comparable, and for which model? |
| **Transforms** | is this column the wrong shape, and does fixing it help? |

And one constraint runs through all of them, from 04-05: **every one of these steps is *fitted*, and a
fitted step is fitted on training data only.** 04-07 makes that automatic; this chapter makes it
meaningful.

## What you will be able to do

- Read a missingness map, and test whether missingness is informative
- Choose an imputation strategy, and add the indicator column when it earns its place
- Avoid the comparison trap that makes dropping rows look best when it is not
- Encode a category without inventing an order, and price what an arbitrary order costs
- Say which models need scaling and which are indifferent, with numbers
- Log a skewed column, and know what the back-transform does to your predictions

## Warm-up: retrieve, do not reread

1. Which preprocessing steps can leak the target, and which cannot?
2. In 03-07, which methods were unaffected by scaling?
3. In 02-04, what made a sentinel value like `-1` more dangerous than a genuine `NaN`?

<br>

*Answers: (1) steps that look at `y` - target encoding, feature selection - can; steps that are functions
of `X` alone - scalers, plain imputers, one-hot - cannot. (2) trees, forests and boosting. (3) it is silently
included in every average, so it corrupts results without producing an error.*

## The data

City rental listings. Twelve districts, and the district codes are administrative - **`D00` is not
cheaper or dearer than `D07`, the numbering means nothing.** That detail matters later.

| Column | Meaning |
|---|---|
| `district` | one of twelve administrative codes |
| `area_m2` | floor area, **13% missing** |
| `rooms` | number of rooms |
| `age_years` | age of the building, **13% missing** |
| `has_lift` | 1 if the building has a lift |
| `rent_eur` | monthly rent - the target |

In [ ]:
import warnings

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

warnings.filterwarnings("ignore")


# SYNTHETIC. One row per rental listing in a city with twelve districts.
def load_flats():
    rng = np.random.default_rng(19)
    n = 1200
    levels = ["D%02d" % i for i in range(12)]
    district = rng.choice(levels, n, p=np.array([9, 8, 7, 6, 6, 5, 5, 4, 4, 3, 2, 1]) / 60)
    # district codes are administrative, so their order says nothing about price
    premium = dict(zip(rng.permutation(levels), np.linspace(1.35, 0.75, 12)))

    area = np.round(np.exp(rng.normal(4.0, 0.35, n)), 1)
    rooms = np.clip(np.round(area / 28 + rng.normal(0, 0.5, n)), 1, 6).astype(int)
    age = np.clip(np.round(rng.gamma(3.0, 14.0, n)), 0, 140).astype(int)
    has_lift = (rng.random(n) < 1 / (1 + np.exp(3 - 0.05 * (120 - age)))).astype(int)
    rent = np.round(np.maximum(150, 6.0 * area * np.array([premium[d] for d in district])
                               - 1.2 * age + 60 * has_lift + rng.normal(0, 60, n)), 0)

    flats = pd.DataFrame({"district": district, "area_m2": area, "rooms": rooms,
                          "age_years": age, "has_lift": has_lift, "rent_eur": rent})
    # the age records for OLD buildings were lost, so missingness carries information
    flats.loc[rng.random(n) < 0.05 + 0.45 * (flats.age_years > 60), "age_years"] = np.nan
    # area is missing for no reason at all
    flats.loc[rng.random(n) < 0.12, "area_m2"] = np.nan
    return flats


flats = load_flats()
NUMERIC = ["area_m2", "rooms", "age_years", "has_lift"]
CATEGORICAL = ["district"]
target = flats.rent_eur

print("%d listings, %d columns" % flats.shape)
print(flats.isna().sum().rename("missing").to_frame().T.to_string())

## Decision 1 · Missing values

### First look at where the holes are

Before choosing what to put in them, look at the pattern. A **missingness map** - one pixel per cell,
coloured by whether it is present - shows in one glance whether the holes are scattered or structured.

In [ ]:
fig, (left, right) = plt.subplots(1, 2, figsize=(12.5, 4.6),
                                  gridspec_kw={"width_ratios": [2, 1]})

sample = flats.head(200)
left.imshow(sample.isna().T.to_numpy(), aspect="auto", cmap="Blues", interpolation="nearest")
left.set_yticks(range(flats.shape[1]))
left.set_yticklabels(flats.columns, fontsize=9)
left.set_xlabel("listing (first 200)")
left.set_title("Missingness map: dark means the value is absent", fontsize=11)

shares = 100 * flats.isna().mean()
bars = right.barh(range(len(shares)), shares.to_numpy(),
                  color=["#D55E00" if v > 0 else "#cccccc" for v in shares])
for position, value in enumerate(shares):
    if value > 0:
        right.text(value + 0.3, position, "%.1f%%" % value, va="center", fontsize=9)
right.set_yticks(range(len(shares)))
right.set_yticklabels(shares.index, fontsize=9)
right.set_xlim(0, 18)
right.set_xlabel("percentage missing")
right.set_title("Two columns have holes", fontsize=11)
right.invert_yaxis()

plt.tight_layout()
plt.show()

### The question that decides everything: is the missingness informative?

**A missing value is a value.** Sometimes it means "we did not measure this", and sometimes it means
something about the row. The test is simple: **compare the target between rows where the value is present
and rows where it is not.**

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12.5, 4.3), sharey=True)
for ax, column in zip(axes, ["age_years", "area_m2"]):
    absent = flats[column].isna()
    ax.hist(target[~absent], bins=30, alpha=0.75, color="#0072B2", density=True, label="value present")
    ax.hist(target[absent], bins=30, alpha=0.75, color="#D55E00", density=True, label="value missing")
    ax.axvline(target[~absent].mean(), color="#0072B2", linewidth=2)
    ax.axvline(target[absent].mean(), color="#D55E00", linewidth=2)
    ax.set_xlabel("rent (EUR)")
    ax.set_title("%s   -   means %.0f and %.0f, a gap of %.0f"
                 % (column, target[~absent].mean(), target[absent].mean(),
                    target[absent].mean() - target[~absent].mean()), fontsize=10.5)
    ax.legend(fontsize=8)
axes[0].set_ylabel("density")
fig.suptitle("One of these columns has informative missingness. The other does not", fontsize=12, y=1.02)
plt.tight_layout()
plt.show()

for column in ["age_years", "area_m2"]:
    absent = flats[column].isna()
    print("%-10s missing in %4d rows | mean rent %.0f present, %.0f missing | gap %+.0f"
          % (column, absent.sum(), target[~absent].mean(), target[absent].mean(),
             target[absent].mean() - target[~absent].mean()))

**`age_years` is missing in a way that means something: those flats rent for 33 EUR less on average.**
`area_m2` is missing for no reason connected to the rent - a gap of 14 EUR, small next to a spread of
this size.

That is not an accident of this dataset, it is how it was built: the age records **for old buildings**
were lost. So "age is missing" is a coded message saying "this building is old", and old buildings are
cheaper. The information is in the *hole*, not in the value.

**This is the distinction that decides the imputation strategy**, and it has standard names:

| Pattern | Means | What to do |
|---|---|---|
| **Missing completely at random** | the hole is unrelated to anything | impute; the choice barely matters |
| **Missing at random** | the hole depends on *other columns* you have | impute using those columns |
| **Missing not at random** | the hole depends on the **unseen value itself** | impute **and add an indicator column**; be honest that you cannot fully recover it |

`age_years` here is the third kind - the worst kind - because whether the value is missing depends on
what the value would have been. **No imputation can recover it**, and pretending otherwise is the mistake.
What you *can* do is stop throwing away the one piece of information that survived: the fact that it is
missing.

### What mean imputation actually does

Filling a hole with the column mean is not neutral. Here is the shape it leaves behind.

In [ ]:
from sklearn.impute import SimpleImputer

filled_area = flats.area_m2.fillna(flats.area_m2.mean())
was_missing = flats.area_m2.isna()

fig, (left, right) = plt.subplots(1, 2, figsize=(12.5, 4.4))

left.scatter(flats.area_m2[~was_missing], target[~was_missing], s=10, alpha=0.3,
             color="#0072B2", label="observed")
left.set_xlabel("area (m2)")
left.set_ylabel("rent (EUR)")
left.set_title("Before: %d rows have no area at all" % was_missing.sum(), fontsize=11)
left.legend(fontsize=8)

right.scatter(filled_area[~was_missing], target[~was_missing], s=10, alpha=0.3,
              color="#0072B2", label="observed")
right.scatter(filled_area[was_missing], target[was_missing], s=14, alpha=0.6,
              color="#D55E00", label="filled with the mean")
right.axvline(flats.area_m2.mean(), color="#D55E00", linestyle="--")
right.set_xlabel("area (m2)")
right.set_title("After: a vertical stripe of invented flats", fontsize=11)
right.legend(fontsize=8)

plt.tight_layout()
plt.show()

print("the imputed rows now all claim an area of %.1f m2" % flats.area_m2.mean())
print("their rents still range from %.0f to %.0f EUR"
      % (target[was_missing].min(), target[was_missing].max()))

**The orange stripe is 156 flats that now claim to be exactly average-sized**, while their rents still
vary from one end of the range to the other. Mean imputation has told the model that area does not
predict rent for these rows - which will pull the fitted relationship flatter.

It also **shrinks the column's variance** and **distorts its correlation with everything else**. None of
that makes it wrong; it makes it a choice with consequences, and the consequences are worth measuring
rather than assuming.

### Measuring the strategies - and the trap

In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.linear_model import Ridge
from sklearn.model_selection import KFold, cross_val_score
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

folds = KFold(5, shuffle=True, random_state=0)


def build(strategy="mean", add_indicator=False):
    numeric = make_pipeline(SimpleImputer(strategy=strategy, add_indicator=add_indicator),
                            StandardScaler())
    return make_pipeline(
        ColumnTransformer([("numeric", numeric, NUMERIC),
                           ("categorical", OneHotEncoder(handle_unknown="ignore"), CATEGORICAL)]),
        Ridge())


def error_of(model, frame, labels):
    return -cross_val_score(model, frame, labels, cv=folds,
                            scoring="neg_mean_absolute_error").mean()


complete = flats[NUMERIC].notna().all(axis=1)

rows = [
    {"strategy": "mean", "rows used": len(flats),
     "MAE": round(error_of(build("mean"), flats[NUMERIC + CATEGORICAL], target), 2)},
    {"strategy": "median", "rows used": len(flats),
     "MAE": round(error_of(build("median"), flats[NUMERIC + CATEGORICAL], target), 2)},
    {"strategy": "mean + missing indicator", "rows used": len(flats),
     "MAE": round(error_of(build("mean", True), flats[NUMERIC + CATEGORICAL], target), 2)},
    {"strategy": "drop incomplete rows", "rows used": int(complete.sum()),
     "MAE": round(error_of(build("mean"), flats.loc[complete, NUMERIC + CATEGORICAL],
                           target[complete]), 2)},
]
print(pd.DataFrame(rows).to_string(index=False))

**Dropping the incomplete rows scores 51.41 and everything else scores about 58.** If you stopped here you
would conclude that discarding a quarter of your data is the best available strategy.

It is not. **The four numbers are not comparable, because the last one is measured on different rows.**

In [ ]:
same_rows = flats.loc[complete, NUMERIC + CATEGORICAL]
same_target = target[complete]

print("judged on the SAME 915 complete rows:")
print("  drop incomplete rows : MAE %.2f" % error_of(build("mean"), same_rows, same_target))
print("  mean imputation      : MAE %.2f" % error_of(build("mean"), same_rows, same_target))
print()
print("they are the same procedure on those rows - there is nothing left to impute.")
print()
print("mean rent, complete rows %.0f   vs   all rows %.0f"
      % (same_target.mean(), target.mean()))
print("mean rent of the rows that dropping discards: %.0f" % target[~complete].mean())

**Identical, because on complete rows there is nothing to impute.** The entire apparent advantage of
dropping rows was that **it changed the question to an easier one.**

The discarded rows are the old buildings whose age records were lost - cheaper, and harder to predict.
Removing them makes the remaining problem easier, and the MAE falls for a reason that has nothing to do
with the merits of dropping data.

> **A metric is only comparable across procedures that are answering the same question on the same rows.**
> Whenever a preprocessing choice changes which rows survive, the score changes for two reasons at once,
> and only one of them is the one you were investigating.

This is the same trap as 04-04's E10, where a stricter split was also a smaller one. It is worth having a
name for, because it recurs constantly: **any step that filters rows makes its own evaluation look
better.** In production those rows still arrive, and the model must predict something for them.

### The indicator column earns its place

Back on the honest comparison - all 1,200 rows - **mean plus an indicator scores 57.89 against mean
alone at 58.50.** A small gain, and it comes entirely from `age_years`, which is exactly where the
informative-missingness test said it would.

In [ ]:
rows = []
for label, columns in [("indicator for both columns", NUMERIC),
                       ("indicator for age only", ["age_years"]),
                       ("indicator for area only", ["area_m2"])]:
    frame = flats[NUMERIC + CATEGORICAL].copy()
    for column in columns:
        frame[column + "_missing"] = flats[column].isna().astype(int)
    extra = [c for c in frame.columns if c.endswith("_missing")]
    numeric = make_pipeline(SimpleImputer(), StandardScaler())
    model = make_pipeline(
        ColumnTransformer([("numeric", numeric, NUMERIC + extra),
                           ("categorical", OneHotEncoder(handle_unknown="ignore"), CATEGORICAL)]),
        Ridge())
    rows.append({"added": label, "MAE": round(error_of(model, frame, target), 2)})
rows.append({"added": "nothing (mean imputation only)",
             "MAE": round(error_of(build("mean"), flats[NUMERIC + CATEGORICAL], target), 2)})
print(pd.DataFrame(rows).to_string(index=False))

**The `age_years` indicator does the work and the `area_m2` indicator does nothing** - which is the
prediction the histograms made, now confirmed by the model.

The habit that follows is cheap and general: **add the indicator for any column whose missingness passed
the informative test, and let the model decide what to do with it.** It costs one binary column. If the
missingness turns out to be uninformative, a regularised model will give it a coefficient near zero and
you have lost nothing.

## Decision 2 · Encoding a category

`district` is twelve words. A model needs numbers. There are three standard answers and they are not
interchangeable.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(13.5, 3.9))
example = ["D03", "D07", "D03", "D11"]

axes[0].set_title("ordinal\none column, an invented order", fontsize=10.5)
for row, value in enumerate(example):
    axes[0].add_patch(plt.Rectangle((0.1, -row), 1.0, 0.8, facecolor="#cfe3f3", edgecolor="white",
                                    linewidth=2))
    axes[0].text(0.6, -row + 0.4, value, ha="center", va="center", fontsize=10)
    axes[0].annotate("", xy=(1.55, -row + 0.4), xytext=(1.15, -row + 0.4),
                     arrowprops=dict(arrowstyle="-|>", color="#888888"))
    axes[0].add_patch(plt.Rectangle((1.6, -row), 0.7, 0.8, facecolor="#f6d3bd", edgecolor="white",
                                    linewidth=2))
    axes[0].text(1.95, -row + 0.4, str(int(value[1:])), ha="center", va="center", fontsize=10)
axes[0].set_xlim(0, 2.6)

axes[1].set_title("one-hot\ntwelve columns, no order", fontsize=10.5)
for row, value in enumerate(example):
    axes[1].add_patch(plt.Rectangle((0.1, -row), 1.0, 0.8, facecolor="#cfe3f3", edgecolor="white",
                                    linewidth=2))
    axes[1].text(0.6, -row + 0.4, value, ha="center", va="center", fontsize=10)
    for slot in range(12):
        on = slot == int(value[1:])
        axes[1].add_patch(plt.Rectangle((1.35 + slot * 0.28, -row), 0.24, 0.8,
                                        facecolor="#D55E00" if on else "#eeeeee",
                                        edgecolor="white", linewidth=1))
axes[1].set_xlim(0, 4.8)

axes[2].set_title("target encoding\none column, uses y - see 04-05", fontsize=10.5)
pretend = {"D03": 412, "D07": 298, "D11": 355}
for row, value in enumerate(example):
    axes[2].add_patch(plt.Rectangle((0.1, -row), 1.0, 0.8, facecolor="#cfe3f3", edgecolor="white",
                                    linewidth=2))
    axes[2].text(0.6, -row + 0.4, value, ha="center", va="center", fontsize=10)
    axes[2].annotate("", xy=(1.55, -row + 0.4), xytext=(1.15, -row + 0.4),
                     arrowprops=dict(arrowstyle="-|>", color="#888888"))
    axes[2].add_patch(plt.Rectangle((1.6, -row), 1.0, 0.8, facecolor="#f3d6e3", edgecolor="white",
                                    linewidth=2))
    axes[2].text(2.1, -row + 0.4, str(pretend[value]), ha="center", va="center", fontsize=10)
axes[2].set_xlim(0, 3.2)

for ax in axes:
    ax.set_ylim(-3.4, 1.0)
    ax.set_xticks([])
    ax.set_yticks([])
    for side in ax.spines.values():
        side.set_visible(False)
plt.tight_layout()
plt.show()

**Ordinal** turns `D07` into 7. That is one column and it is cheap, and it tells the model that `D07` is
seven times `D01` and sits between `D06` and `D08`. **For an administrative code, all of that is false.**

**One-hot** makes one column per district, each 0 or 1. No order is implied, and the model learns twelve
independent adjustments. The cost is twelve columns instead of one.

**Target encoding** replaces the code with the average rent in that district. One column, and it captures
the ordering that actually matters - at the price of 04-05's leak if it is not computed inside the fold.

**Predict before running:** how much does an arbitrary ordinal order cost, against one-hot?

In [ ]:
from sklearn.preprocessing import OrdinalEncoder

numeric = make_pipeline(SimpleImputer(), StandardScaler())
rows = []
for label, encoder, columns in [
        ("one-hot, 12 columns", OneHotEncoder(handle_unknown="ignore"), CATEGORICAL),
        ("ordinal, 1 column, arbitrary order", OrdinalEncoder(), CATEGORICAL),
        ("district dropped entirely", None, None)]:
    if encoder is None:
        model = make_pipeline(ColumnTransformer([("numeric", numeric, NUMERIC)]), Ridge())
    else:
        model = make_pipeline(
            ColumnTransformer([("numeric", numeric, NUMERIC), ("categorical", encoder, columns)]),
            Ridge())
    rows.append({"encoding": label,
                 "MAE": round(error_of(model, flats[NUMERIC + CATEGORICAL], target), 2)})
encoding_table = pd.DataFrame(rows)
print(encoding_table.to_string(index=False))

best, worst = encoding_table.MAE.min(), encoding_table.MAE.max()
ordinal = encoding_table.MAE.iloc[1]
print()
print("of the %.2f EUR that district information is worth, an arbitrary order recovers %.2f - %.1f%%"
      % (worst - best, worst - ordinal, 100 * (worst - ordinal) / (worst - best)))

In [ ]:
fig, ax = plt.subplots(figsize=(9, 4.2))
positions = range(len(encoding_table))
bars = ax.bar(positions, encoding_table.MAE, color=["#0072B2", "#D55E00", "#999999"], width=0.62)
for position, value in zip(positions, encoding_table.MAE):
    ax.text(position, value + 0.6, "%.2f" % value, ha="center", fontsize=10)
ax.axhline(best, color="#0072B2", linestyle="--", linewidth=1.2)
ax.axhline(worst, color="#999999", linestyle="--", linewidth=1.2)
ax.annotate("", xy=(1, best), xytext=(1, worst),
            arrowprops=dict(arrowstyle="<->", color="#000000", linewidth=1.4))
ax.text(1.08, (best + worst) / 2, "all the information\nthe district carries",
        fontsize=9, va="center")
ax.set_xticks(positions)
ax.set_xticklabels(encoding_table.encoding.str.replace(", ", ",\n"), fontsize=9)
ax.set_ylabel("mean absolute error (EUR)")
ax.set_ylim(0, 90)
ax.set_title("An arbitrary order throws away most of what the column knows", fontsize=11.5)
plt.tight_layout()
plt.show()

**Ordinal encoding recovers 9.5% of what the district column is worth.** Dropping it entirely costs 77.75;
one-hot gets to 58.50; the arbitrary order gets to 75.92 - barely better than not having the column at
all.

The reason is worth seeing clearly: a linear model given an ordinal code fits **one coefficient** for it,
so it can only express "rent rises steadily with the district number". The district numbers were shuffled,
so no such pattern exists, and one straight line through twelve unordered groups is close to a flat line.

**When ordinal encoding is right:** when the order is real - `small < medium < large`, `never < sometimes
< often`, a star rating. Then one column captures it and twelve would be wasteful.

**When one-hot becomes awkward:** high cardinality. Twelve districts is twelve columns; ten thousand
postcodes is ten thousand, which is wide data with all of 04-05's hazards. That is where target encoding
earns its keep - **and where it is most dangerous**, because high cardinality means few rows per category.
Both facts have the same cause.

### The category that was not in the training data

One more thing that only shows up in production.

In [ ]:
train_flats = flats[flats.district != "D11"]
new_listing = flats[flats.district == "D11"].head(3)

strict = make_pipeline(
    ColumnTransformer([("numeric", numeric, NUMERIC),
                       ("categorical", OneHotEncoder(handle_unknown="error"), CATEGORICAL)]),
    Ridge()).fit(train_flats[NUMERIC + CATEGORICAL], train_flats.rent_eur)

forgiving = make_pipeline(
    ColumnTransformer([("numeric", numeric, NUMERIC),
                       ("categorical", OneHotEncoder(handle_unknown="ignore"), CATEGORICAL)]),
    Ridge()).fit(train_flats[NUMERIC + CATEGORICAL], train_flats.rent_eur)

try:
    strict.predict(new_listing[NUMERIC + CATEGORICAL])
except ValueError as failure:
    print("handle_unknown='error'  ->", str(failure).split("\n")[0][:96])

print("handle_unknown='ignore' -> predictions %s"
      % np.round(forgiving.predict(new_listing[NUMERIC + CATEGORICAL]), 0))
print("                           actual rents %s" % new_listing.rent_eur.to_numpy())

**A district the model has never seen crashes one version and is silently treated as "none of the above"
by the other.** Neither behaviour is automatically right:

- `handle_unknown="error"` fails loudly, which is what you want in a batch job you can retry, and
  catastrophic in a live service.
- `handle_unknown="ignore"` encodes the unknown district as all-zeros - **the model falls back to its
  intercept plus whatever the numeric columns say**, which is a reasonable degraded prediction and a
  completely silent one.

**The choice belongs in the framing, not in the modelling**, and the question is: when an unfamiliar
value arrives, would you rather be wrong or be stopped? A monitoring counter on the number of unknown
categories is worth more than either.

## Decision 3 · Scaling

03-07 and 03-08 both argued for standardising, from different directions - distance is measured in the
largest unit, and an optimiser's step size has to suit every direction at once. Here is the same claim
tested on a real pipeline, across three models.

**Predict before running:** which of ridge, k-nearest-neighbours and a random forest will change?

In [ ]:
from sklearn.ensemble import RandomForestRegressor
from sklearn.neighbors import KNeighborsRegressor

rows = []
for name, model in [("ridge regression", Ridge()),
                    ("k-nearest neighbours", KNeighborsRegressor(10)),
                    ("random forest", RandomForestRegressor(200, random_state=0, min_samples_leaf=2))]:
    scaled = make_pipeline(
        ColumnTransformer([("numeric", make_pipeline(SimpleImputer(), StandardScaler()), NUMERIC),
                           ("categorical", OneHotEncoder(handle_unknown="ignore"), CATEGORICAL)]),
        model)
    unscaled = make_pipeline(
        ColumnTransformer([("numeric", SimpleImputer(), NUMERIC),
                           ("categorical", OneHotEncoder(handle_unknown="ignore"), CATEGORICAL)]),
        model)
    with_scaling = error_of(scaled, flats[NUMERIC + CATEGORICAL], target)
    without = error_of(unscaled, flats[NUMERIC + CATEGORICAL], target)
    rows.append({"model": name, "scaled": round(with_scaling, 2), "unscaled": round(without, 2),
                 "cost of not scaling": round(without - with_scaling, 2)})
scaling_table = pd.DataFrame(rows)
print(scaling_table.to_string(index=False))

In [ ]:
fig, (left, right) = plt.subplots(1, 2, figsize=(12.5, 4.4))

spread = flats[NUMERIC].agg(["min", "max"]).T
left.barh(range(len(spread)), spread["max"] - spread["min"], color="#0072B2")
for position, (name, row) in enumerate(spread.iterrows()):
    left.text(row["max"] - row["min"] + 2, position, "%g to %g" % (row["min"], row["max"]),
              va="center", fontsize=9)
left.set_yticks(range(len(spread)))
left.set_yticklabels(spread.index, fontsize=9)
left.set_xlim(0, 200)
left.set_xlabel("range of the column, in its own units")
left.set_title("The columns are not comparable", fontsize=11)
left.invert_yaxis()

positions = np.arange(len(scaling_table))
right.barh(positions + 0.19, scaling_table.unscaled, height=0.36, color="#D55E00", label="unscaled")
right.barh(positions - 0.19, scaling_table.scaled, height=0.36, color="#0072B2", label="scaled")
for position, row in scaling_table.iterrows():
    right.text(row.unscaled + 0.8, position + 0.19, "%+.2f" % row["cost of not scaling"],
               va="center", fontsize=9)
right.set_yticks(positions)
right.set_yticklabels(scaling_table.model.str.replace(" ", "\n"), fontsize=8.5)
right.set_xlim(0, 95)
right.set_xlabel("mean absolute error (EUR)")
right.set_title("Only one of the three notices", fontsize=11)
right.legend(fontsize=8, loc="lower right")

plt.tight_layout()
plt.show()

**k-nearest neighbours pays 15.65 EUR for not scaling. Ridge pays -0.01 and the forest pays -0.01** -
which is to say nothing at all, twice.

Exactly as 03-07 predicted, and it is worth being precise about *why each*:

- **kNN** computes distances. `age_years` spans 0 to 140 and `has_lift` spans 0 to 1, so without scaling
  the age difference decides every neighbour and the lift is invisible.
- **The forest** splits one column at a time and only ever asks "is this value above that threshold?",
  which is unchanged by any monotone rescaling.
- **Ridge** is more subtle. A linear model's *fit* is unaffected by scaling - it just adjusts the
  coefficients - but its **penalty** is not, because the penalty treats all coefficients alike. Here the
  effect is too small to see; on data with wilder ranges, or a larger penalty, it is not. **Scale linear
  models with regularisation** even though this table says you got away with it.

The practical rule stays what it was: **scale by default, know why, and know the exceptions.**

## Decision 4 · Transforming a column

`rent_eur` is right-skewed, as prices usually are. 03-06 said that logging a skewed variable makes
multiplicative relationships additive. Does it help here?

In [ ]:
from sklearn.model_selection import cross_val_predict

fig, (left, right) = plt.subplots(1, 2, figsize=(12.5, 4.1))
left.hist(target, bins=40, color="#0072B2")
left.axvline(target.mean(), color="#D55E00", linewidth=2)
left.axvline(target.median(), color="#000000", linewidth=2, linestyle="--")
left.set_xlabel("rent (EUR)")
left.set_ylabel("listings")
left.set_title("rent: mean %.0f, median %.0f - a right tail" % (target.mean(), target.median()),
               fontsize=11)

right.hist(np.log(target), bins=40, color="#009E73")
right.set_xlabel("log(rent)")
right.set_title("log(rent): the tail is pulled in", fontsize=11)
plt.tight_layout()
plt.show()

pipeline = make_pipeline(
    ColumnTransformer([("numeric", make_pipeline(SimpleImputer(), StandardScaler()), NUMERIC),
                       ("categorical", OneHotEncoder(handle_unknown="ignore"), CATEGORICAL)]),
    Ridge())

direct = cross_val_predict(pipeline, flats[NUMERIC + CATEGORICAL], target, cv=folds)
in_logs = cross_val_predict(pipeline, flats[NUMERIC + CATEGORICAL], np.log(target), cv=folds)
back = np.exp(in_logs)
smearing = float(np.exp(np.log(target) - in_logs).mean())

print("%-42s %8s %12s" % ("", "MAE", "mean prediction"))
print("%-42s %8.2f %12.1f" % ("predict rent directly", np.abs(target - direct).mean(), direct.mean()))
print("%-42s %8.2f %12.1f" % ("predict log(rent), then exp back",
                              np.abs(target - back).mean(), back.mean()))
print("%-42s %8.2f %12.1f" % ("...with a smearing correction of %.4f" % smearing,
                              np.abs(target - back * smearing).mean(), (back * smearing).mean()))
print("%-42s %8s %12.1f" % ("the truth", "-", target.mean()))

**Logging made it slightly worse here - 59.25 against 58.50 - and the back-transform is biased low.**

Two separate findings, and the second is the one people are not warned about.

**Logging did not help** because the relationship in this data is mostly additive already - rent is built
from area, age and a lift, added together. The log transform is for data where effects *multiply*, and
imposing it where they do not costs accuracy. **A transform is a hypothesis about the shape of the
relationship, not a tidying step.**

**The back-transform is biased.** `exp(mean of logs)` is the geometric mean, which is always **below** the
arithmetic mean - so exponentiating a prediction of `log(rent)` systematically under-predicts `rent`. The
mean prediction lands at 352.5 against a true mean of 358.1 - short by 5.6 EUR. That is not noise; it is a
property of the
transform, and it is the same fact 03-06 met as "the geometric mean is not the mean".

**Duan's smearing estimator** fixes the level by multiplying by `mean(exp(residual))` - here 1.0248 - and
the mean prediction becomes 361.2, overshooting slightly rather than undershooting. Note that it repairs
most of the *bias* and slightly worsens the *MAE* (59.25 to 59.47), which
is a genuine trade: if you are summing predictions into a revenue forecast, take the correction; if you
are minimising per-listing error, do not.

**The rule to carry away:** if you model a transformed target, you have changed what you are predicting,
and getting back to the original units is a second modelling decision with its own error.

## Putting it together

Four decisions, one summary.

In [ ]:
fig, ax = plt.subplots(figsize=(11.5, 5.2))
summary = [
    ("Imputation", "#cfe3f3",
     "test whether the missingness is informative, then impute + add an indicator",
     "age indicator earned 0.61 EUR; area's earned nothing"),
    ("Encoding", "#f6d3bd",
     "one-hot unless the order is real; target-encode high cardinality, inside the fold",
     "an arbitrary ordinal order recovered 9.5% of the column's value"),
    ("Scaling", "#cfe8dc",
     "scale by default; it decides distance methods and is free for the rest",
     "kNN paid 15.65 EUR unscaled; ridge and the forest paid nothing"),
    ("Transforms", "#f3d6e3",
     "a transform is a hypothesis, and the back-transform is a second decision",
     "logging cost 0.75 EUR and under-predicted the mean by 5.6"),
]
for position, (title, colour, rule, evidence) in enumerate(summary):
    y = len(summary) - position - 1
    ax.add_patch(plt.Rectangle((0.05, y + 0.06), 1.75, 0.88, facecolor=colour, edgecolor="white",
                               linewidth=3))
    ax.text(0.93, y + 0.5, title, ha="center", va="center", fontsize=12, fontweight="bold")
    ax.text(1.95, y + 0.62, rule, va="center", fontsize=10, color="#222222")
    ax.text(1.95, y + 0.28, evidence, va="center", fontsize=8.5, color="#777777", style="italic")
ax.set_xlim(0, 9.4)
ax.set_ylim(-0.1, len(summary) + 0.35)
ax.set_xticks([])
ax.set_yticks([])
for side in ax.spines.values():
    side.set_visible(False)
ax.set_title("Four preprocessing decisions, and what each one was worth here", fontsize=13)
plt.tight_layout()
plt.show()

Every number in that figure is modest - between nothing and 16 EUR on rents averaging 358. **That is
worth saying, because preprocessing is often discussed as though it were where models are won.** On this
data the district encoding was worth 19 EUR, the missingness indicator 0.6, and scaling mattered for
exactly one of three models.

The one big number is the district encoding, worth 19.25 EUR - and that is a *feature* decision wearing
preprocessing clothes. What preprocessing reliably does is **lose** you performance when it is done
thoughtlessly - an arbitrary ordinal code, a transform imposed on the wrong shape, a dropped quarter of
the data justified by an incomparable metric. It is a place to avoid errors rather than to find wins, and the errors are quiet.

## Common misconceptions

**"Imputation fills in the missing information."**
It fills the hole. When the missingness is informative, the information was in the hole, and the indicator
column is the only part you can recover.

**"Dropping incomplete rows is the clean, honest option."**
It changed the score from 58.50 to 51.41 by removing the hard rows. On the same rows it is identical to
mean imputation, because there is nothing left to impute.

**"Ordinal encoding is a compact one-hot."**
It asserts an order. Here that cost 90% of the column's value.

**"One-hot is always right for categories."**
Twelve levels, yes. Ten thousand postcodes, no - that is wide data, with 04-05's hazards.

**"Scaling is essential."**
For distance methods it decides the answer. For trees it does nothing measurable, and here for ridge it
did nothing either. Know which of the three you have.

**"Logging a skewed variable is good practice."**
It is a hypothesis that effects multiply. Here they add, and logging cost 0.75 EUR and biased the level.

**"I can transform the target and just exponentiate the predictions."**
That systematically under-predicts, because `exp` of a mean of logs is a geometric mean.

## Exercises

Solutions: `solutions/04_workflow/04-06_preprocessing_solutions.ipynb`.

### Quick understanding

**E1.** Name the three missingness patterns and say what each implies for imputation.

**E2.** Why is a metric computed after dropping incomplete rows not comparable with one computed after
imputing them?

**E3.** Which of the four preprocessing decisions can leak the target, and which cannot?

### Hand calculation

**E4.** A column has values 10, 20, 30, missing, missing. Compute the mean and the standard deviation
before and after mean-imputing. State what happened to the spread, and why that matters for a distance
method.

**E5.** A category has three levels encoded ordinally as 1, 2, 3, with true effects of +100, -50 and +30
on the target. Explain why a linear model with one coefficient cannot represent them, and give the number
of coefficients one-hot would need.

**E6.** `log(rent)` predictions for three flats are 5.6, 6.0 and 6.4. Compute `exp` of each and their mean.
Then compute the mean of the actual rents if the true log-rents were 5.5, 6.1 and 6.4. Which is larger,
and does that match the chapter's claim?

**E7.** A postcode column has 8,000 levels in 50,000 rows. Compute the average rows per level. Using
04-05's E4 rule, say whether target encoding is safe, and how many columns one-hot would add.

### Coding

**E8.** Write `missingness_report(frame, target)` that, for every column with holes, reports the share
missing and the difference in mean target between missing and present rows, sorted by the size of the
difference. Run it on `flats`.

**E9.** Replace `SimpleImputer` with `KNNImputer` and with `IterativeImputer`. Do the cleverer imputers
beat the mean here? Report the numbers and say what that suggests about effort spent on imputation.

**E10.** Build the ordinal encoding again, but order the districts by their **training-fold** mean rent
instead of alphabetically. Report the MAE. How close does one column get to one-hot's twelve, and what
have you just re-invented?

**E11.** Write a `ColumnTransformer` pipeline that does all four decisions correctly and cross-validate
it. Then deliberately fit the scaler outside and confirm, using 04-05's result, that it barely matters -
and say why you would still not do it.

**E12.** The chapter used `handle_unknown="ignore"`. Write a wrapper that instead **counts** unseen
categories at predict time and warns when the share exceeds a threshold. Test it on the held-out `D11`.

### Interpretation

**E13.** A colleague reports that imputing with a random forest improved their model from MAE 58 to 41.
Give the first question you would ask.

**E14.** Your model uses target encoding on a `customer_id` column with one row per customer. Explain in
two sentences what it has learned and what it will do in production.

### Debugging

**E15.** After adding one-hot encoding your model's training time went up 40-fold and its score got
slightly worse. Name the likely cause and two fixes.

**E16.** A pipeline scores well in cross-validation and crashes in production with a shape error. Give the
two most likely causes.

### Exam and interview reasoning

**E17.** "How do you handle missing data?" Answer in under a minute, and be ready for: "why not just drop
those rows?"

### Transfer to a different situation

**E18.** You are modelling hospital readmission. Columns include `blood_pressure` (missing 30%),
`ward_code` (60 levels), `days_since_last_admission` (missing when there was no previous admission), and
`cost_eur` (heavily skewed). Give your preprocessing decision for each, and say which of the four
missingness patterns `days_since_last_admission` has.

### Explain it to someone non-technical

**E19.** Explain in under 90 words why "we filled the gaps with the average" is not a neutral act, using
an everyday example.

### Optional challenge

**E20.** Build the **informative-missingness demonstration from scratch**: generate a dataset where a
column is missing completely at random, and another identical dataset where it is missing depending on its
own value. Fit the same model on both, with and without indicator columns, and show that the indicator
earns its place in exactly one of the four combinations. This is the cleanest way to prove to yourself
that the *pattern*, not the *rate*, is what matters.

In [ ]:
# Your workspace. In memory: flats, NUMERIC, CATEGORICAL, target, folds, build,
# error_of, numeric, complete, pipeline, encoding_table, scaling_table.

## Mastery check

- [ ] Read a missingness map and test whether missingness is informative
- [ ] Name the three missingness patterns and pick a strategy for each
- [ ] Explain why dropping rows flatters its own evaluation
- [ ] Choose between ordinal, one-hot and target encoding for a stated column
- [ ] Say which models need scaling, and why each does or does not
- [ ] Decide whether to log a variable, on evidence rather than on skew alone
- [ ] Predict a transformed target and get back to the original units honestly

## What should now feel instinctive

- Looking at the missingness pattern before choosing what to fill holes with
- Distrusting any comparison where the row count changed
- Asking whether a category's order is real before encoding it as a number
- Scaling by default and knowing it is free for trees
- Treating a transform as a claim about the relationship, not as tidying

## Flashcards

| Front | Back |
|---|---|
| MCAR / MAR / MNAR | Unrelated to anything / depends on other columns / depends on the missing value itself |
| Informative missingness test | Compare the target between rows where the value is present and absent |
| Missing indicator | A binary column saying "this was missing". Earns its place when the test says so |
| The dropping trap | Dropping incomplete rows scored 51.41 against 58.50 - by removing the hard rows |
| Same-rows rule | A metric is comparable only across procedures answering the same question on the same rows |
| Ordinal encoding | One column, asserts an order. Recovered 9.5% of the district's value here |
| One-hot encoding | One column per level, no order. Twelve here; unusable at ten thousand |
| Unseen category | `handle_unknown`: crash loudly or predict silently. A framing decision |
| Scaling, by model | kNN paid 15.65 EUR; ridge and random forest paid nothing |
| Why trees ignore scaling | They ask "above this threshold?", which no monotone rescaling changes |
| Logging the target | A hypothesis that effects multiply. Cost 0.75 EUR here |
| Back-transform bias | `exp(mean of logs)` is the geometric mean, always below the arithmetic mean |
| Smearing correction | Multiply by `mean(exp(residual))` - fixes the level, not necessarily the MAE |

## Next

**04-07 · Pipelines and cross-validation without leaking.** This chapter made four fitted decisions and
kept saying "inside the fold". The next one builds the machinery that enforces it: `Pipeline`,
`ColumnTransformer`, and cross-validation that refits every step on every fold.

After that, 04-08 closes the module with reproducibility - seeds, environments, and the experiment record
that lets somebody else get your number back.